## Building A Chatbot
In this We'll go over an example of how to design and implement an LLM-powered chatbot. This chatbot will be able to have a conversation and remember previous interactions.

Note that this chatbot that we build will only use the language model to have a conversation. There are several other related concepts that you may be looking for:

- Conversational RAG: Enable a chatbot experience over an external source of data
- Agents: Build a chatbot that can take actions

This  will cover the basics which will be helpful for those two more advanced topics.

In [26]:
import os
from dotenv import load_dotenv
load_dotenv() ## aloading all the environment variable

groq_api_key=os.getenv("GROQ_API_KEY")


In [27]:
from langchain_groq import ChatGroq
model=ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000013C7B9DDFA0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000013C7B9DE900>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [28]:
from langchain_core.messages import HumanMessage
model.invoke([HumanMessage(content="Hi , My name is Zaid and I am learning GENAI")])

AIMessage(content='Hey Zaid! 👋  \nNice to meet you—welcome to the world of Generative AI!  \n\nWhat’s catching your interest right now? Are you diving into language models, image generation, fine‑tuning, or something else? Let me know what you’re working on (or planning to explore), and I’ll be happy to help with explanations, code snippets, resources, or just a quick sanity check on your ideas. 🚀', additional_kwargs={'reasoning_content': 'The user says "Hi , My name is Zaid and I am learning GENAI". This is a greeting. We should respond friendly, ask about their learning progress, maybe ask what topics they are learning. Also we should mention that we can help with questions. The user hasn\'t asked a specific question. So we can respond with a friendly greeting and ask about what aspects of GENAI they are focusing on. Also mention that we can help with code, explanations, etc.'}, response_metadata={'token_usage': {'completion_tokens': 195, 'prompt_tokens': 84, 'total_tokens': 279, 'co

In [29]:
from langchain_core.messages import AIMessage
model.invoke(
    [
        HumanMessage(content="Hi , My name is Zaid and I am learning GENAI"),
        AIMessage(content="'Hello Zaid! 👋  \nIt’s great to meet someone diving into Generative AI. How can I help you on your learning journey today? Are you looking for resources, project ideas, explanations of concepts, or something else? Let me know what interests you most!"),
        HumanMessage(content="Hey What's my name and what do I do?")
    ]
)

AIMessage(content='You’re Zaid, and you’re learning Generative AI—exploring how models can create text, images, music, and more. 🚀 If you need help with concepts, projects, or resources, just let me know!', additional_kwargs={'reasoning_content': 'The user says "Hey What\'s my name and what do I do?" We should answer that their name is Zaid, and they are learning Generative AI. We should respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 160, 'total_tokens': 254, 'completion_time': 0.104489968, 'completion_tokens_details': {'reasoning_tokens': 38}, 'prompt_time': 0.007796088, 'prompt_tokens_details': None, 'queue_time': 0.050309072, 'total_time': 0.112286056}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5c8ca06ea1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dda4e-929c-7c82-9185-3b30823a4905-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'in

### Message History
We can use a Message History class to wrap our model and make it stateful. This will keep track of inputs and outputs of the model, and store them in some datastore. Future interactions will then load those messages and pass them into the chain as part of the input. Let's see how to use this!

In [30]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store={}

def get_session_history(session_id:str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id]=ChatMessageHistory()
    return store[session_id]

with_message_history=RunnableWithMessageHistory(model,get_session_history)

In [31]:
config={"configurable":{"session_id":"chat1"}}

In [32]:
response=with_message_history.invoke(
    [HumanMessage(content="Hi , My name is Zaid and I am learning GENAI")],
    config=config
)

In [33]:
response.content

'Hello Zaid! 👋  \nIt’s great to hear that you’re diving into Generative AI. How can I support your learning journey today? Whether you’re looking for resources, project ideas, or just a quick explanation of a concept, I’m here to help!'

In [34]:
with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

AIMessage(content='Your name is Zaid.', additional_kwargs={'reasoning_content': 'User asks: "What\'s my name?" They gave earlier: "Hi , My name is Zaid and I am learning GENAI". So answer: Zaid.'}, response_metadata={'token_usage': {'completion_tokens': 49, 'prompt_tokens': 153, 'total_tokens': 202, 'completion_time': 0.053495816, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.007345524, 'prompt_tokens_details': None, 'queue_time': 0.04977764, 'total_time': 0.06084134}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_5c8ca06ea1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dda4e-9508-7c71-8946-69fa50d617df-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 153, 'output_tokens': 49, 'total_tokens': 202, 'output_token_details': {'reasoning': 34}})

In [35]:
## change the config-->session id
config1={"configurable":{"session_id":"chat2"}}
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'I don’t have that information. Could you tell me what you’d like me to call you?'

In [36]:
response=with_message_history.invoke(
    [HumanMessage(content="Hey My name is azeem")],
    config=config1
)
response.content

'Nice to meet you, Azeem! How can I help you today?'

In [37]:
response=with_message_history.invoke(
    [HumanMessage(content="Whats my name")],
    config=config1
)
response.content

'Your name is Azeem.'

### Prompt templates
Prompt Templates help to turn raw user information into a format that the LLM can work with. In this case, the raw user input is just a message, which we are passing to the LLM. Let's now make that a bit more complicated. First, let's add in a system message with some custom instructions (but still taking messages as input). Next, we'll add in more input besides just the messages.

In [38]:
from langchain_core.prompts import ChatPromptTemplate,MessagesPlaceholder
prompt=ChatPromptTemplate.from_messages(
    [
        ("system","You are a helpful assistant.Amnswer all the question to the best of your ability"),
        MessagesPlaceholder(variable_name="messages")
    ]
)

chain=prompt|model

In [39]:
chain.invoke({"messages":[HumanMessage(content="Hi My name is zaid")]})

AIMessage(content='Hello Zaid! 👋 How can I assist you today?', additional_kwargs={'reasoning_content': 'User says "Hi My name is zaid". We need to respond politely, maybe ask how we can help. The instructions: "You are a helpful assistant. Answer all the question to the best of your ability." There\'s no question yet, so we can greet and ask how to help.'}, response_metadata={'token_usage': {'completion_tokens': 82, 'prompt_tokens': 98, 'total_tokens': 180, 'completion_time': 0.088346035, 'completion_tokens_details': {'reasoning_tokens': 60}, 'prompt_time': 0.005438454, 'prompt_tokens_details': None, 'queue_time': 0.04995012, 'total_time': 0.093784489}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_c5a89987dc', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dda4e-9920-7380-b0f1-3137cbbf7b0e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 82, 'total_tokens': 

In [40]:
with_message_history=RunnableWithMessageHistory(chain,get_session_history)

In [41]:
config = {"configurable": {"session_id": "chat3"}}
response=with_message_history.invoke(
    [HumanMessage(content="Hi My name is zaid")],
    config=config
)

response

AIMessage(content='Hello Zaid! 👋 How can I help you today?', additional_kwargs={'reasoning_content': 'User says: "Hi My name is zaid". Probably wants a greeting. Provide friendly response.'}, response_metadata={'token_usage': {'completion_tokens': 43, 'prompt_tokens': 98, 'total_tokens': 141, 'completion_time': 0.047086929, 'completion_tokens_details': {'reasoning_tokens': 21}, 'prompt_time': 0.005469612, 'prompt_tokens_details': None, 'queue_time': 0.049255973, 'total_time': 0.052556541}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_80501ff3a1', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019dda4e-9a2b-77e3-b2e8-f392885f9b38-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 98, 'output_tokens': 43, 'total_tokens': 141, 'output_token_details': {'reasoning': 21}})

In [42]:
response = with_message_history.invoke(
    [HumanMessage(content="What's my name?")],
    config=config,
)

response.content

'Your name is Zaid.'

In [43]:
## Add more complexity

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer all questions to the best of your ability in {language}.",
        ),
        MessagesPlaceholder(variable_name="messages"),
    ]
)

chain = prompt | model

In [44]:
response=chain.invoke({"messages":[HumanMessage(content="Hi My name is Zaid")],"language":"hindi"})
response.content 

'नमस्ते ज़ैद! आपका स्वागत है। आपको कैसे मदद कर सकता हूँ?'

Let's now wrap this more complicated chain in a Message History class. This time, because there are multiple keys in the input, we need to specify the correct key to use to save the chat history.

In [45]:
with_message_history=RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages"
)

In [46]:
config = {"configurable": {"session_id": "chat4"}}
repsonse=with_message_history.invoke(
    {'messages': [HumanMessage(content="Hi,I am Zaid")],"language":"Hindi"},
    config=config
)
repsonse.content

'नमस्ते ज़ैद! आपका स्वागत है। मैं आपकी किस प्रकार सहायता कर सकता हूँ?'

In [47]:
response = with_message_history.invoke(
    {"messages": [HumanMessage(content="whats my name?")], "language": "Hindi"},
    config=config,
)

In [48]:
response.content

'आपका नाम ज़ैद है।'

### Managing the Conversation History
One important concept to understand when building chatbots is how to manage conversation history. If left unmanaged, the list of messages will grow unbounded and potentially overflow the context window of the LLM. Therefore, it is important to add a step that limits the size of the messages you are passing in.
'trim_messages' helper to reduce how many messages we're sending to the model. The trimmer allows us to specify how many tokens we want to keep, along with other parameters like if we want to always keep the system message and whether to allow partial messages

In [49]:

from langchain_core.messages import SystemMessage,trim_messages
trimmer=trim_messages(
    max_tokens=45,
    strategy="last",
    token_counter=model,
    include_system=True,
    allow_partial=False,
    start_on="human"
)
messages = [
    SystemMessage(content="you're a good assistant"),
    HumanMessage(content="hi! I'm zaid"),
    AIMessage(content="hi!"),
    HumanMessage(content="I like vanilla ice cream"),
    AIMessage(content="nice"),
    HumanMessage(content="whats 2 + 2"),
    AIMessage(content="4"),
    HumanMessage(content="thanks"),
    AIMessage(content="no problem!"),
    HumanMessage(content="having fun?"),
    AIMessage(content="yes!"),
]
trimmer.invoke(messages)

[SystemMessage(content="you're a good assistant", additional_kwargs={}, response_metadata={}),
 HumanMessage(content='I like vanilla ice cream', additional_kwargs={}, response_metadata={}),
 AIMessage(content='nice', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='whats 2 + 2', additional_kwargs={}, response_metadata={}),
 AIMessage(content='4', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='thanks', additional_kwargs={}, response_metadata={}),
 AIMessage(content='no problem!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]),
 HumanMessage(content='having fun?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='yes!', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]

In [50]:
from operator import itemgetter

from langchain_core.runnables import RunnablePassthrough

chain=(
    RunnablePassthrough.assign(messages=itemgetter("messages")|trimmer)
    | prompt
    | model
    
)

response=chain.invoke(
    {
    "messages":messages + [HumanMessage(content="What ice cream do i like")],
    "language":"English"
    }
)
response.content

'I’m not sure—could you tell me a bit about what flavors or textures you usually enjoy? For example, do you like classic vanilla or something more adventurous like salted caramel or a fruity sorbet? That way I can suggest something that might hit the spot!'

In [51]:
response = chain.invoke(
    {
        "messages": messages + [HumanMessage(content="what math problem did i ask")],
        "language": "English",
    }
)
response.content

'You asked: “What’s 2\u202f+\u202f2?”'

In [52]:
## Lets wrap this in the MEssage History
with_message_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="messages",
)
config={"configurable":{"session_id":"chat5"}}

In [53]:
response = with_message_history.invoke(
    {
        "messages": messages + [HumanMessage(content="whats my name?")],
        "language": "English",
    },
    config=config,
)

response.content

'I don’t actually have a record of your name—unless you tell me!'

In [54]:
response = with_message_history.invoke(
    {
        "messages": [HumanMessage(content="what math problem did i ask?")],
        "language": "English",
    },
    config=config,
)

response.content

'I’m not sure—so far you haven’t asked me any math problem. If you have one in mind, just let me know and I’ll be happy to help!'